# PiT-Mem (PVLDB): 双时态金融 Agent 记忆存储与索引引擎 — Colab Pro 一键复现主控台

本 Notebook 对应论文 **PiT-Mem: A Point-in-Time Bi-Temporal Storage and Indexing Engine for Look-Ahead-Free Agentic Memory** 的完整单卡 A100 High-RAM 实验流水线（`Exp 1 ~ Exp 6`）。

### 核心架构与实验覆盖：
1. **高速本地 NVMe 冷热分层与持久化评测 (`Exp 6B/6C/6D`)**：`1,032 B/rec` 核心索引常驻内存 + `3,072 B/rec` 原始 FP32 向量 POSIX `mmap` 按需分页卸载（`DONTNEED` 冷页缺失压测）、原子 `mmap` CSR 快照与 IEEE 802.3 CRC32 WAL 恢复、`PhaseFairSharedMutex` 12 线程并发读 + 持续流式写入/重述/宏段滚动压测，以及 16GB `shared_buffers` 热驻留 PostgreSQL (`pgvector v0.8.0`) 均运行在 `/content/local_ssd/` 本地 NVMe 上。
2. **431,217 条真实去重金融语料 + 分片断点续跑**：自动拉取 ALFRED 宏观多版本流、SEC EDGAR `10-K/10-Q/8-K Item 4.02` 重述流与 39.4 万条去重金融披露文本，按 20 万条/分片完成 200 万条 `BAAI/bge-base-en-v1.5` (768d FP16) 向量编码，并审计跨轨迹多样性与同谱系演化余弦相似度。
3. **全场景对比基线 (`Exp 1 ~ Exp 6`)**：覆盖 Workload 1A (`12%` 凹函数多段束分配高召回因果图区间) 与 Workload 1B (`1%` 零倒挂解析代价路由区间)、`Grid-SQ8-Scan`、`ACORN-BiTemp (SIGMOD'24)`、`PG-BiTemp (2M Warm RAM)`、`SeRF-Adapt`、`DSG-1D`、`Zep/Graphiti`、端到端 Write+Read Token U 型曲线与 3 跳密集重述风暴自适应 `k_fetch` 倍增 (`Exp 3/3B`)、双协议去偏审计 (`Exp 4`)、`Parquet-Zstd` / `SQLite-BTree` / 2025 后真实 SEC XBRL 披露及 `query_tiered_pit` 冷热跨层统一检索 (`Exp 5/5C`)，以及 Vitter 滞后卷积水库 + 全桶 `>20%` 倾斜守卫在线流式滚动 (`Exp 6A`)。

In [ ]:
# Step 1: 挂载 Google Drive 并将项目高速同步至本地 NVMe (/content/local_ssd)
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/PiT-Mem_VLDB_Project'
LOCAL_ROOT = '/content/local_ssd/pit-mem'
os.makedirs('/content/local_ssd', exist_ok=True)
if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(DRIVE_ROOT, LOCAL_ROOT, ignore=shutil.ignore_patterns('*.so', 'build'))
%cd /content/local_ssd/pit-mem
!nvidia-smi
!lscpu | head -n 18 && free -h

In [ ]:
# Step 2: 安装 PostgreSQL + pgvector v0.8.0、C++17/OpenMP 依赖并编译 pit_mem_core (AVX2 + FMA)
!sudo apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq build-essential cmake pybind11-dev postgresql postgresql-contrib postgresql-server-dev-all zstd texlive-latex-extra texlive-fonts-extra
!sudo apt-get install -y -qq postgresql-16-pgvector || sudo apt-get install -y -qq postgresql-14-pgvector || true
!pip install -q --upgrade pybind11 psycopg[binary] sentence-transformers transformers accelerate pyarrow pandas matplotlib scikit-learn
!pip install --no-build-isolation ./cpp
!PYTHONPATH=. python3 -m pit_mem.benchmarks.test_mw1_mw4_fixes
import pit_mem_core as pm
print('Compiled & verified pit_mem_core successfully! INF_TICK =', pm.INF_TICK)

In [ ]:
# Step 3: 运行 200 万规模全套基准实验（Exp 1 ~ Exp 6）
!PYTHONUNBUFFERED=1 PYTHONPATH=. python3 -m pit_mem.benchmarks.run_all_experiments \
    --mode colab_pro \
    --work-dir /content/local_ssd/work_2m \
    --raw-seed-dir data/raw_seed \
    --out-dir results/colab_pro \
    --dim 768 \
    --llm-model Qwen/Qwen2.5-7B-Instruct

In [ ]:
# Step 4: 生成 PVLDB 出版级矢量 PDF 图表与 LaTeX 实测指标宏，并同步回 Google Drive
!PYTHONPATH=. python3 -m pit_mem.plotting.generate_figures results/colab_pro paper/figures
!PYTHONPATH=. python3 -m pit_mem.plotting.generate_latex_macros --results-dir results/colab_pro --out-tex-dir paper

import shutil
shutil.copytree('results/colab_pro', f'{DRIVE_ROOT}/results/colab_pro', dirs_exist_ok=True)
shutil.copytree('paper/figures', f'{DRIVE_ROOT}/paper/figures', dirs_exist_ok=True)
shutil.copy2('paper/generated_metrics.tex', f'{DRIVE_ROOT}/paper/generated_metrics.tex')
shutil.copy2('paper/generated_tables.tex', f'{DRIVE_ROOT}/paper/generated_tables.tex')
print('All Colab Pro results & vector figures synced back to Google Drive:', DRIVE_ROOT)